In [2]:
import sys
sys.path.append('/data5/users/dahee/Causal_Path/CausalPathTracing_for_ViT/main')
import numpy as np
import timm
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as models
import numpy as np
import matplotlib.pyplot as plt
import random
import json
import urllib.request
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import networkx as nx
from collections import defaultdict
import matplotlib
import numpy as np
import torch
from timm import create_model
from fvcore.nn import FlopCountAnalysis, flop_count_table
from lib.models import ModelAndTokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer

/home/cwkang/miniconda3/envs/EAP/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [30]:
def union_dict(causal_path):
    union_dict_ = {}
    for key, value in causal_path.items():
        # value is a list of lists
        union_set = set()
        for sublist in value:
            union_set.update(sublist)
        union_dict_[key] = sorted(union_set)
    return union_dict_

def select_module_vision(union_dict_):
    selected_module = []
    selected_module.extend(['patch_embed', 'pos_drop', 'patch_drop', 'norm_pre','norm','fc_norm','head_drop','head'])
    selected_module_attn = []

    for key, value in union_dict_.items():
        if any(x in value for x in [1,5,6,7]):
            selected_module.append(f'blocks.{key}.mlp')
            selected_module.append(f'blocks.{key}.ls2')
            selected_module.append(f'blocks.{key}.drop_path2')
            selected_module.append(f'blocks.{key}.norm2')
        if any(x in value for x in [2,5]):
            selected_module_attn.append(f'blocks.{key}.attn')
            selected_module_attn.append(f'blocks.{key}.norm1')
            selected_module_attn.append(f'blocks.{key}.ls1')
            selected_module_attn.append(f'blocks.{key}.drop_path1')
        if any(x in value for x in [3,6]):
            selected_module_attn.append(f'blocks.{key}.attn')
            selected_module_attn.append(f'blocks.{key}.norm1')
            selected_module_attn.append(f'blocks.{key}.ls1')
            selected_module_attn.append(f'blocks.{key}.drop_path1')
        if any(x in value for x in [4,7]):
            selected_module_attn.append(f'blocks.{key}.attn')
            selected_module_attn.append(f'blocks.{key}.norm1')
            selected_module_attn.append(f'blocks.{key}.ls1')
            selected_module_attn.append(f'blocks.{key}.drop_path1')
    return selected_module, selected_module_attn

def selected_module_gptxs(union_dict_, n_head):
    selected_module = []
    selected_module.extend(['transformer.wte','transformer.wpe','transformer.drop','transformer.ln_f','lm_head'])
    selected_module_attn = []


    for key, value in union_dict_.items():
        mlp_key = [1] + list(np.arange(n_head + 2, n_head * 2 + 2))
        if any(x in value for x in mlp_key):
            selected_module.append(f'transformer.h.{key}.mlp')
            selected_module.append(f'transformer.h.{key}.ln_2')

        for i in range(n_head):
            attn_key = [i+2,i+2+n_head]
            if any(x in value for x in attn_key):
                selected_module_attn.append(f'transformer.h.{key}.attn')
                selected_module_attn.append(f'transformer.h.{key}.ln_1')
    return selected_module, selected_module_attn

def selected_module_pythia(union_dict_, n_head):
    selected_module = []
    selected_module.extend(['model.gpt_neox.embed_in', 'model.gpt_neox.emb_dropout','model.gpt_neox.final_layer_norm','model.gpt_neox.rotary_emb','model.embed_out'])
    selected_module_attn = []


    for key, value in union_dict_.items():
        if any(x in value for x in [1]):
            selected_module.append(f'model.gpt_neox.layers.{key}.mlp')
            selected_module.append(f'model.gpt_neox.layers.{key}.post_mlp_dropout')
            selected_module_attn.append(f'model.gpt_neox.layers.{key}.post_attention_layernorm')
        for i in range(n_head):
            attn_key = list(np.arange(2,n_head+2))
            if any(x in value for x in attn_key):
                selected_module_attn.append(f'model.gpt_neox.layers.{key}.attention')
                selected_module_attn.append(f'model.gpt_neox.layers.{key}.input_layernorm')
                selected_module_attn.append(f'model.gpt_neox.layers.{key}.post_attention_dropout')
    return selected_module, selected_module_attn

## ViT-T

In [31]:
import json


model = create_model('vit_tiny_patch16_224', pretrained=False)
model.eval()


x = torch.randn(1, 3, 224, 224)
n_head = 3


flops = FlopCountAnalysis(model, x)

for dataset in ["imagenet", "officehome"]:
    print(dataset)
    for z in ["samplewise_0.25", "classwise_0.25", "taskwise_0.25"]:
        print(z)
        result_folder = f'/data8/cwkang/workspace/edge_attribution_patching/jobs_EAP_IG/{dataset}_vit_tiny_patch16_224_{z}/results'

        flops_list = []
        for curr_dir in sorted(os.listdir(result_folder)):
            if curr_dir == "log_error_samples.txt" or curr_dir == "path_forward_results":
                continue
            # curr_result = os.path.join(result_folder, curr_dir, "results")
            curr_result = os.path.join(result_folder, curr_dir)
            # idx = int(curr_dir.split("_")[1])
            idx = int(curr_dir[1:])
            # json_path = os.path.join(curr_result, "R{:04d}".format(idx), "C{:06d}.json".format(idx))
            json_path = os.path.join(curr_result, "C{:06d}.json".format(idx))

            if not os.path.isfile(json_path):
                continue  # Skip if the result JSON file does not exist
            with open(json_path, "r") as f:
                causal_path = json.load(f)

            union_dict_ = union_dict(causal_path)
            selected_module, selected_module_attn = select_module_vision(union_dict_)

            selected_flops_mlp = 0
            for name in selected_module:
                if name in flops.by_module():
                    selected_flops_mlp += flops.by_module()[name]

            selected_flops_attn = 0
            for name in selected_module_attn:
                if name in flops.by_module():
                    selected_flops_attn += flops.by_module()[name]
            
            selected_flops = selected_flops_mlp + selected_flops_attn/n_head
            flops_list.append(selected_flops/flops.total())
        print(np.mean(flops_list))

imagenet
samplewise_0.25


Unsupported operator aten::add encountered 25 time(s)
Unsupported operator aten::scaled_dot_product_attention encountered 12 time(s)
Unsupported operator aten::gelu encountered 12 time(s)
The following submodules of the model were never called during the trace of the graph. They may be unused, or they were accessed by direct calls to .forward() or via other python methods. In the latter case they will have zeros for statistics, though their statistics will still contribute to their parent calling module.
blocks.0.attn.attn_drop, blocks.1.attn.attn_drop, blocks.10.attn.attn_drop, blocks.11.attn.attn_drop, blocks.2.attn.attn_drop, blocks.3.attn.attn_drop, blocks.4.attn.attn_drop, blocks.5.attn.attn_drop, blocks.6.attn.attn_drop, blocks.7.attn.attn_drop, blocks.8.attn.attn_drop, blocks.9.attn.attn_drop


0.8604596965840497
classwise_0.25
0.9136773368485643
taskwise_0.25
0.981944848799476
officehome
samplewise_0.25
0.8560825447195263
classwise_0.25
0.8976389532911725
taskwise_0.25
0.9729172731992144


# DeiT-T

In [32]:
import json

model = create_model('deit_tiny_patch16_224', pretrained=False)
model.eval()


x = torch.randn(1, 3, 224, 224)
n_head = 3


flops = FlopCountAnalysis(model, x)

for dataset in ["imagenet", "officehome"]:
    print(dataset)
    for z in ["samplewise_0.25", "classwise_0.25", "taskwise_0.25"]:
        print(z)
        result_folder = f'/data8/cwkang/workspace/edge_attribution_patching/jobs_EAP_IG/{dataset}_deit_tiny_patch16_224_{z}/results'

        flops_list = []
        for curr_dir in sorted(os.listdir(result_folder)):
            if curr_dir == "log_error_samples.txt" or curr_dir == "path_forward_results":
                continue
            # curr_result = os.path.join(result_folder, curr_dir, "results")
            curr_result = os.path.join(result_folder, curr_dir)
            # idx = int(curr_dir.split("_")[1])
            idx = int(curr_dir[1:])
            # json_path = os.path.join(curr_result, "R{:04d}".format(idx), "C{:06d}.json".format(idx))
            json_path = os.path.join(curr_result, "C{:06d}.json".format(idx))

            if not os.path.isfile(json_path):
                continue  # Skip if the result JSON file does not exist
            with open(json_path, "r") as f:
                causal_path = json.load(f)

            union_dict_ = union_dict(causal_path)
            selected_module, selected_module_attn = select_module_vision(union_dict_)

            selected_flops_mlp = 0
            for name in selected_module:
                if name in flops.by_module():
                    selected_flops_mlp += flops.by_module()[name]

            selected_flops_attn = 0
            for name in selected_module_attn:
                if name in flops.by_module():
                    selected_flops_attn += flops.by_module()[name]
            
            selected_flops = selected_flops_mlp + selected_flops_attn/n_head
            flops_list.append(selected_flops/flops.total())
        print(np.mean(flops_list))

imagenet
samplewise_0.25


Unsupported operator aten::add encountered 25 time(s)
Unsupported operator aten::scaled_dot_product_attention encountered 12 time(s)
Unsupported operator aten::gelu encountered 12 time(s)
The following submodules of the model were never called during the trace of the graph. They may be unused, or they were accessed by direct calls to .forward() or via other python methods. In the latter case they will have zeros for statistics, though their statistics will still contribute to their parent calling module.
blocks.0.attn.attn_drop, blocks.1.attn.attn_drop, blocks.10.attn.attn_drop, blocks.11.attn.attn_drop, blocks.2.attn.attn_drop, blocks.3.attn.attn_drop, blocks.4.attn.attn_drop, blocks.5.attn.attn_drop, blocks.6.attn.attn_drop, blocks.7.attn.attn_drop, blocks.8.attn.attn_drop, blocks.9.attn.attn_drop


0.9548945941204655
classwise_0.25
0.9648493956850536
taskwise_0.25
0.9638896975989524
officehome
samplewise_0.25
0.9247898749042391
classwise_0.25
0.9463668844071063
taskwise_0.25
0.9819448487994761


In [33]:
print('all')
print(round((0.981944848799476 + 0.9729172731992144 + 0.9638896975989524 + 0.9819448487994761) / 4, 4))
print()

print('cls')
print(round((0.9136773368485643 + 0.8976389532911725 + 0.9648493956850536 + 0.9463668844071063) / 4, 4))
print()

print('samplewise')
print(round((0.8604596965840497 + 0.8560825447195263 + 0.9548945941204655 + 0.9247898749042391) / 4, 4))

all
0.9752

cls
0.9306

samplewise
0.8991


## GPT-XS

In [26]:
model_name = "AlgorithmicResearchGroup/gpt2-xs"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

# 2. Prepare dummy input
prompt = "Hello, this is a test input."
inputs = tokenizer(prompt, return_tensors="pt")

# 3. Trace the model on input and measure FLOPs
# We trace only the 'input_ids' part
input_ids = inputs["input_ids"]
n_head = model.config.n_head
# Some versions may require disabling cache
with torch.no_grad():
    flops = FlopCountAnalysis(model, (input_ids,))

result_folder = '/data5/users/dahee/causal_path/250512_collection/20250505_0612_debug_gpt2-xs_exStWd'
result_folder = '/data5/users/dahee/causal_path/250512_collection/20250505_0612_debug_gpt2-xs_lama_trex_exStWd'

flops_list = []

'''아래 result.json 파일 가져오는 코드 수정'''
results_path = os.path.join(result_folder, "results")
for curr_result in sorted(os.listdir(results_path)):
    idx = int(curr_result.split("R")[-1])
    json_path = os.path.join(results_path, curr_result, "C{:06d}.json".format(idx))
    f = open(json_path,"r")
    causal_path = json.load(f)
    '''result.json 파일 가져오는 코드 수정'''

    union_dict_ = union_dict(causal_path)
    selected_module, selected_module_attn = selected_module_gptxs(union_dict_, n_head)

    selected_flops_mlp = 0
    for name in selected_module:
        if name in flops.by_module():
            selected_flops_mlp += flops.by_module()[name]

    selected_flops_attn = 0
    for name in selected_module_attn:
        if name in flops.by_module():
            selected_flops_attn += flops.by_module()[name]
    
    selected_flops = selected_flops_mlp + selected_flops_attn/n_head
    flops_list.append(selected_flops/flops.total())
print(np.mean(flops_list))

Unsupported operator aten::add encountered 26 time(s)
Unsupported operator aten::embedding encountered 2 time(s)
Unsupported operator aten::pow encountered 12 time(s)
Unsupported operator aten::div encountered 6 time(s)
Unsupported operator aten::sub encountered 6 time(s)
Unsupported operator aten::where encountered 6 time(s)
Unsupported operator aten::softmax encountered 6 time(s)
Unsupported operator aten::mul encountered 24 time(s)
Unsupported operator aten::tanh encountered 6 time(s)


0.8929493689556683


# PYTHIA

In [27]:
# Load model and tokenizer
model_name = "EleutherAI/pythia-1b"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

# Prepare input
prompt = "Hello world!"
inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs["input_ids"]

# Wrap model to avoid unsupported DynamicCache object
class WrappedModel(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids):
        # Force disable use_cache
        return self.model(input_ids=input_ids, use_cache=False).logits

wrapped_model = WrappedModel(model)
n_head = 8

# Measure FLOPs
with torch.no_grad():
    flops = FlopCountAnalysis(wrapped_model, (input_ids,))

result_folder = '/data5/users/dahee/causal_path/250512_collection/20250505_0612_debug_pythia-1b_exStWd'
result_folder = "/data5/users/dahee/causal_path/250512_collection/20250505_0612_debug_pythia-1b_lama_trex_exStWd"

flops_list = []

'''아래 result.json 파일 가져오는 코드 수정'''
results_path = os.path.join(result_folder, "results")
for curr_result in sorted(os.listdir(results_path)):
    idx = int(curr_result.split("R")[-1])
    json_path = os.path.join(results_path, curr_result, "C{:06d}.json".format(idx))
    f = open(json_path,"r")
    causal_path = json.load(f)
    '''result.json 파일 가져오는 코드 수정'''


    union_dict_ = union_dict(causal_path)
    selected_module, selected_module_attn = selected_module_pythia(union_dict_, n_head)

    selected_flops_mlp = 0
    for name in selected_module:
        if name in flops.by_module():
            selected_flops_mlp += flops.by_module()[name]

    selected_flops_attn = 0
    for name in selected_module_attn:
        if name in flops.by_module():
            selected_flops_attn += flops.by_module()[name]
    
    selected_flops = selected_flops_mlp + selected_flops_attn/n_head
    flops_list.append(selected_flops/flops.total())

print(np.mean(flops_list))

Unsupported operator aten::embedding encountered 1 time(s)
Unsupported operator aten::add encountered 83 time(s)
Unsupported operator aten::triu encountered 1 time(s)
Unsupported operator aten::mul_ encountered 1 time(s)
Unsupported operator aten::cos encountered 1 time(s)
Unsupported operator aten::mul encountered 82 time(s)
Unsupported operator aten::sin encountered 1 time(s)
Unsupported operator aten::neg encountered 32 time(s)
Unsupported operator aten::softmax encountered 16 time(s)
Unsupported operator aten::gelu encountered 16 time(s)


0.6966196435039743


In [28]:
# Load model and tokenizer
model_name = "EleutherAI/pythia-14m" 
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

# Prepare input
prompt = "Hello world!"
inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs["input_ids"]

# Wrap model to avoid unsupported DynamicCache object
class WrappedModel(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids):
        # Force disable use_cache
        return self.model(input_ids=input_ids, use_cache=False).logits

wrapped_model = WrappedModel(model)
n_head = 4

# Measure FLOPs
with torch.no_grad():
    flops = FlopCountAnalysis(wrapped_model, (input_ids,))

result_folder = '/data5/users/dahee/causal_path/250512_collection/20250505_0612_debug_pythia-14m_exStWd'
result_folder = "/data5/users/dahee/causal_path/250512_collection/20250505_0612_debug_pythia-14m_lama_trex_exStWd"

flops_list = []

'''아래 result.json 파일 가져오는 코드 수정'''
results_path = os.path.join(result_folder, "results")
for curr_result in sorted(os.listdir(results_path)):
    idx = int(curr_result.split("R")[-1])
    json_path = os.path.join(results_path, curr_result, "C{:06d}.json".format(idx))
    f = open(json_path,"r")
    causal_path = json.load(f)
    '''result.json 파일 가져오는 코드 수정'''


    union_dict_ = union_dict(causal_path)
    selected_module, selected_module_attn = selected_module_pythia(union_dict_, n_head)

    selected_flops_mlp = 0
    for name in selected_module:
        if name in flops.by_module():
            selected_flops_mlp += flops.by_module()[name]

    selected_flops_attn = 0
    for name in selected_module_attn:
        if name in flops.by_module():
            selected_flops_attn += flops.by_module()[name]
    
    selected_flops = selected_flops_mlp + selected_flops_attn/n_head
    flops_list.append(selected_flops/flops.total())

print(np.mean(flops_list))

Unsupported operator aten::embedding encountered 1 time(s)
Unsupported operator aten::add encountered 33 time(s)
Unsupported operator aten::triu encountered 1 time(s)
Unsupported operator aten::mul_ encountered 1 time(s)
Unsupported operator aten::cos encountered 1 time(s)
Unsupported operator aten::mul encountered 32 time(s)
Unsupported operator aten::sin encountered 1 time(s)
Unsupported operator aten::neg encountered 12 time(s)
Unsupported operator aten::softmax encountered 6 time(s)
Unsupported operator aten::gelu encountered 6 time(s)


0.9850672808925273
